# Глава 29. Работа с текстом до LLM: Bag of Words и TF-IDF

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь превратить текст в числа методами Bag of Words и TF-IDF, обучить классический классификатор текста на sklearn и объяснить, почему на смену этим методам пришли эмбеддинги.

Между классическими моделями (фаза 2) и большими языковыми моделями (фаза 4) есть важный мост, который стоит пройти. Как классифицировать текст, не имея доступа к мощной LLM? Как вообще машина работала с текстом до эпохи трансформеров? Ответ — превратить слова в числа простыми и понятными способами. Это фундамент, на котором стоит всё понимание обработки языка.

**Аналогия: подсчёт слов как отпечаток текста  💡**

Представь, что о содержании текста можно судить по тому, какие слова и сколько раз в нём встречаются. В отзыве со словами «отлично», «супер», «рекомендую» — явно позитив. В отзыве с «ужас», «сломано», «верните» — негатив. Bag of Words («мешок слов») именно так и работает: он высыпает все слова текста в мешок, забыв про порядок, и считает, сколько раз каждое встретилось. Этот «портрет по словам» и есть числовое представление текста.

## Bag of Words: считаем слова

Модель не понимает текст, но понимает числа. Bag of Words превращает каждый документ в строку чисел: сколько раз в нём встретилось каждое слово из общего словаря. Порядок слов теряется — остаётся только их количество.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

тексты = [
    "отличный товар рекомендую",
    "ужасный товар не рекомендую",
    "отличный сервис быстро",
]

vec = CountVectorizer()
X = vec.fit_transform(тексты)

print(vec.get_feature_names_out())  # словарь всех слов
print(X.toarray())                  # каждый текст -> строка чисел

_Каждый текст стал строкой чисел: сколько раз встретилось каждое слово словаря._

## TF-IDF: умный подсчёт

**Аналогия: редкое слово говорит больше  💡**

В Bag of Words слово «товар» из примера встречается везде и потому мало что различает. А слово «ужасный» — редкое и очень показательное. TF-IDF учитывает это: он повышает вес слов, которые часты в конкретном тексте, но редки во всех текстах вместе. Частые везде слова («и», «в», «товар») получают низкий вес, редкие и характерные — высокий. Так важные слова выходят на первый план.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer()
X = vec.fit_transform(тексты)
# Теперь числа — не просто счётчики, а веса важности слов

## Классификатор отзывов на sklearn

Соединим TF-IDF с моделью из фазы 2 в один Pipeline — получится классический классификатор тональности, работающий без всякой LLM. Быстро, дёшево, полностью офлайн.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

отзывы = ["отлично супер рекомендую", "ужас сломано верните деньги",
          "прекрасный товар доволен", "кошмар не берите", "быстро качественно"]
метки  = [1, 0, 1, 0, 1]   # 1 = позитив, 0 = негатив

# TF-IDF + модель в одном конвейере
model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression()),
])
model.fit(отзывы, метки)

print(model.predict(["отличный сервис рекомендую"]))  # -> [1]
print(model.predict(["ужасное качество верните"]))    # -> [0]

Заметь: Pipeline снова спасает — TF-IDF настраивается только на train, без утечки. Это тот же принцип, что и в главе 10, только вместо масштабирования чисел — превращение текста в числа.

## Локальный пример: сортировка обращений граждан

Госпортал или банк в Таджикистане получает тысячи текстовых обращений: жалобы, вопросы, благодарности. TF-IDF + классификатор может автоматически разложить их по категориям и срочности — быстро, офлайн, без затрат на LLM. Для многих реальных задач такой классический подход достаточен и предпочтителен из-за скорости, дешевизны и приватности (данные не уходят в чужой сервис).

## Почему победили эмбеддинги

**Мост к главе 17  ℹ️**

У Bag of Words и TF-IDF есть слабое место: они не понимают смысл. Для них «отличный» и «прекрасный» — два совершенно разных, никак не связанных слова, а «не плохой» они вообще не поймут (порядок и отрицание теряются). Эмбеддинги (глава 17) решили это: они кодируют смысл, и синонимы оказываются рядом на «карте смыслов». Именно поэтому современные системы перешли на эмбеддинги и трансформеры. Но понимать TF-IDF важно: он прост, быстр, работает офлайн и до сих пор незаменим во многих задачах.

## Задания

Базовый уровень

Собери TF-IDF + LogisticRegression на 10 своих отзывах (5 позитивных, 5 негативных). Проверь на новых фразах. Ошибается ли модель? На каких словах она «спотыкается»?

Со звёздочкой  ⭐

Сравни Bag of Words (CountVectorizer) и TF-IDF на одном наборе отзывов по кросс-валидации. Затем придумай фразу с отрицанием («не плохой товар») и проверь — понимает ли её модель? Объясни, почему классические методы тут ошибаются, а эмбеддинги — нет.

Вопросы для обсуждения  💬

Что теряет Bag of Words, «забывая» порядок слов? Приведи пример, где это критично.

Почему редкое слово часто информативнее частого? Как это отражает TF-IDF?

В каких задачах классический TF-IDF предпочтительнее мощной LLM?

**Проверь себя  ℹ️**

1. Как Bag of Words превращает текст в числа?

2. Чем TF-IDF умнее простого подсчёта слов?

3. Почему эмбеддинги вытеснили TF-IDF в современных системах?